In [ ]:
import os, re, math, collections
import pandas as pd
import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4',
            'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(pkg, quiet=True)
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer, PorterStemmer

In [ ]:
if not (os.path.exists('train.csv') and os.path.exists('test.csv')):
    from google.colab import files
    files.upload()

train_df = pd.read_csv('train.csv')
print('train:', train_df.shape)
display(train_df.head())

class_names = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}
print(train_df['Class Index'].map(class_names).value_counts())


Saving train.csv to train.csv
train: (120000, 3)


,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


Class Index
Business    30000
Sci/Tech    30000
Sports      30000
World       30000
Name: count, dtype: int64


In [ ]:
from google.colab import files
uploaded = files.upload()
test_df = pd.read_csv('test.csv')
print("Test Shape:", test_df.shape)
display(test_df.head())

Saving test.csv to test.csv
Test Shape: (7600, 3)


,Class Index,Title,Description
0,3,Fears for T N pension after talks,Unions representing workers at Turner Newall...
1,4,The Race is On: Second Private Team Sets Launc...,"SPACE.com - TORONTO, Canada -- A second\team o..."
2,4,Ky. Company Wins Grant to Study Peptides (AP),AP - A company founded by a chemistry research...
3,4,Prediction Unit Helps Forecast Wildfires (AP),AP - It's barely dawn when Mike Fitzpatrick st...
4,4,Calif. Aims to Limit Farm-Related Smog (AP),AP - Southern California's smog-fighting agenc...


In [ ]:

class_names = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}

print('train:', train_df.shape, '| test:', test_df.shape)
display(train_df.head())
print(train_df['Class Index'].map(class_names).value_counts())
print(test_df['Class Index'].map(class_names).value_counts())

train: (120000, 3) | test: (7600, 3)


,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


Class Index
Business    30000
Sci/Tech    30000
Sports      30000
World       30000
Name: count, dtype: int64
Class Index
Business    1900
Sci/Tech    1900
Sports      1900
World       1900
Name: count, dtype: int64


In [ ]:

for df in (train_df, test_df):
    df['text'] = df['Title'].astype(str) + '. ' + df['Description'].astype(str)
print(train_df['text'][1])

Carlyle Looks Toward Commercial Aerospace (Reuters). Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense industry, has quietly placed\its bets on another part of the market.



 Text preprocessing


In [ ]:
SOURCES = (r'(?:Reuters|AP|AFP|UPI|Associated Press|SPACE\.com|Bloomberg|Canadian Press|'
           r'AP Business|AP Sports|USATODAY\.com|Forbes\.com|PC World|IDG News Service)')

def clean_raw(text):
    t = str(text).replace('\\', ' ')
    t = re.sub(r'&?\blt;.*?&?\bgt;', ' ', t)
    t = re.sub(r'&?\b(?:lt|gt|quot|amp|nbsp);', ' ', t)
    t = re.sub(r'#\d+;', ' ', t)
    t = re.sub(r'(?:https?://|www\.)\S+', ' ', t)
    t = re.sub(r'\bhref\S*|\btarget=\S*', ' ', t)
    t = re.sub(r'\bU\.S\.A?\.?', 'USA', t)
    return t

def clean_title(t):
    t = clean_raw(t).strip()
    return re.sub(r'\(\s*' + SOURCES + r'\s*\)\s*$', '', t).strip()

def clean_desc(d):
    d = clean_raw(d).strip()
    d = re.sub(r'^[^-\u2013\u2014]{0,50}?\(\s*' + SOURCES + r'\s*\)\s*[-\u2013\u2014]+\s*', '', d)
    d = re.sub(r'^' + SOURCES + r'\s*[-\u2013\u2014]+\s*', '', d)
    return d.strip()

def build_text(title, desc):
    title = clean_title(title)
    sep = ' ' if title.endswith(('.', '!', '?')) else '. '
    return title + sep + clean_desc(desc)

for df in (train_df, test_df):
    df['raw_text'] = df['Title'].astype(str) + '. ' + df['Description'].astype(str)
    df['text'] = [build_text(t, d) for t, d in zip(df['Title'], df['Description'])]

print('BEFORE:', train_df['raw_text'][1], '\n')
print('AFTER :', train_df['text'][1])

BEFORE: Carlyle Looks Toward Commercial Aerospace (Reuters). Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense industry, has quietly placed\its bets on another part of the market. 

AFTER : Carlyle Looks Toward Commercial Aerospace. Private investment firm Carlyle Group, which has a reputation for making well-timed and occasionally controversial plays in the defense industry, has quietly placed its bets on another part of the market.


In [ ]:
stop_words = set(stopwords.words('english')) | {'reuters', 'ap', 'afp', 'upi'}
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def wn_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    if tag.startswith('V'): return wordnet.VERB
    if tag.startswith('R'): return wordnet.ADV
    return wordnet.NOUN

def fix_hyphens(t):
    return re.sub(r'\s+', ' ', re.sub(r'[-\u2013\u2014]+', ' ', t)).strip()

i = 1
print('RAW          :', test_df['raw_text'][i], '\n')
s0 = fix_hyphens(test_df['text'][i])
print('CLEANED      :', s0, '\n')

sents = sent_tokenize(s0)
print('SEGMENTATION :')
for k, s in enumerate(sents, 1): print('   ', k, '-', s)
print()

toks = [word_tokenize(s) for s in sents]
print('TOKENIZATION :')
for t in toks: print('   ', t)
print()

tagged = [nltk.pos_tag(t) for t in toks]
print('POS TAGS     :')
for t in tagged: print('   ', t)
print()

lower = [[(w.lower(), tg) for w, tg in t] for t in tagged]
print('LOWERCASE    :')
for t in lower: print('   ', [w for w, _ in t])
print()

no_punct = [[(w, tg) for w, tg in t if w.isalpha() and len(w) > 1] for t in lower]
print('NO PUNCT/NUM :')
for t in no_punct: print('   ', [w for w, _ in t])
print()

no_stop = [[(w, tg) for w, tg in t if w not in stop_words] for t in no_punct]
print('NO STOPWORDS :')
for t in no_stop: print('   ', [w for w, _ in t])
print()

lem = [[lemmatizer.lemmatize(w, wn_pos(tg)) for w, tg in t] for t in no_stop]
stm = [[stemmer.stem(w) for w, _ in t] for t in no_stop]
print('LEMMATIZED   :')
for t in lem: print('   ', t)
print()
print('(STEMMED, for comparison):')
for t in stm: print('   ', t)

RAW          : The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com). SPACE.com - TORONTO, Canada -- A second\team of rocketeers competing for the  #36;10 million Ansari X Prize, a contest for\privately funded suborbital space flight, has officially announced the first\launch date for its manned rocket. 

CLEANED      : The Race is On: Second Private Team Sets Launch Date for Human Spaceflight. TORONTO, Canada A second team of rocketeers competing for the 10 million Ansari X Prize, a contest for privately funded suborbital space flight, has officially announced the first launch date for its manned rocket. 

SEGMENTATION :
    1 - The Race is On: Second Private Team Sets Launch Date for Human Spaceflight.
    2 - TORONTO, Canada A second team of rocketeers competing for the 10 million Ansari X Prize, a contest for privately funded suborbital space flight, has officially announced the first launch date for its manned rocket.

TOKENIZATION :
    ['The', 'R

In [ ]:
def preprocess(text):
    text = fix_hyphens(text)
    out = []
    for sent in sent_tokenize(text):
        tagged = nltk.pos_tag(word_tokenize(sent))
        toks = []
        for w, tg in tagged:
            w = w.lower()
            if not w.isalpha() or len(w) < 2:
                continue
            if w in stop_words:
                continue
            toks.append(lemmatizer.lemmatize(w, wn_pos(tg)))
        if toks:
            out.append(toks)
    return out

val_df        = train_df.sample(frac=0.10, random_state=42)
train_main_df = train_df.drop(val_df.index)

train_sents = [s for txt in train_main_df['text'] for s in preprocess(txt)]
val_sents   = [s for txt in val_df['text']        for s in preprocess(txt)]
test_sents  = [s for txt in test_df['text']       for s in preprocess(txt)]

for name, ss in [('train', train_sents), ('val', val_sents), ('test', test_sents)]:
    print(f'{name:5s} sentences: {len(ss):>8,} | tokens: {sum(map(len, ss)):>10,}')
print(train_sents[:3])

train sentences:  245,400 | tokens:  2,616,810
val   sentences:   27,316 | tokens:    290,916
test  sentences:   17,310 | tokens:    183,702
[['carlyle', 'look', 'toward', 'commercial', 'aerospace'], ['private', 'investment', 'firm', 'carlyle', 'group', 'reputation', 'make', 'well', 'time', 'occasionally', 'controversial', 'play', 'defense', 'industry', 'quietly', 'place', 'bet', 'another', 'part', 'market'], ['oil', 'economy', 'cloud', 'stock', 'outlook']]


## 3. N-gram language model (Bigram, Markov assumption)

### Theory

The chain rule gives the exact probability of a sentence $w_1 \dots w_n$:

$$P(w_1,\dots,w_n)=\prod_{i=1}^{n}P(w_i\mid w_1,\dots,w_{i-1})$$

The **Markov assumption** (first order) says a word depends only on the previous word:

$$P(w_i\mid w_1,\dots,w_{i-1})\approx P(w_i\mid w_{i-1})$$

so

$$P(w_1,\dots,w_n)\approx\prod_{i=1}^{n+1}P(w_i\mid w_{i-1})$$

with the sentence padded as `<s> w1 ... wn </s>` (so $w_0=$ `<s>` and $w_{n+1}=$ `</s>`).

**Maximum likelihood estimate (MLE):**
$$P(w_i\mid w_{i-1})=\frac{C(w_{i-1},w_i)}{C(w_{i-1})}$$

**Add-one (Laplace) smoothing** – needed because an unseen bigram would give probability 0 for the whole sentence:
$$P_{Laplace}(w_i\mid w_{i-1})=\frac{C(w_{i-1},w_i)+1}{C(w_{i-1})+V}$$

where $V$ is the number of different tokens that can be predicted (the vocabulary + `<UNK>` + `</s>`; `<s>` is only ever a context, never predicted).

Build vocabulary and counts — **from TRAIN only**

In [ ]:
MIN_COUNT = 2
word_freq = collections.Counter(w for s in train_sents for w in s)
vocab = {w for w, c in word_freq.items() if c >= MIN_COUNT} | {'<UNK>', '</s>'}
V = len(vocab)
print('vocabulary size V =', V)

def prep(sent):
    return ['<s>'] + [w if w in vocab else '<UNK>' for w in sent] + ['</s>']

unigram  = collections.Counter()
uni_pred = collections.Counter()
bigram   = collections.Counter()
for s in train_sents:
    p = prep(s)
    unigram.update(p)
    uni_pred.update(p[1:])
    bigram.update(zip(p[:-1], p[1:]))
N_pred = sum(uni_pred.values())

followers = collections.defaultdict(list)
for (a, b), c in bigram.items():
    followers[a].append((b, c))

print('distinct bigrams in train:', len(bigram))
print('most common bigrams:', bigram.most_common(10))

vocabulary size V = 32567
distinct bigrams in train: 1044755
most common bigrams: [(('say', '</s>'), 4277), (('<s>', 'new'), 3909), (('new', 'york'), 3590), (('<s>', 'u'), 2982), (('oil', 'price'), 2659), (('<s>', 'microsoft'), 2566), (('<s>', 'usa'), 2336), (('year', '</s>'), 2277), (('united', 'state'), 2273), (('prime', 'minister'), 2228)]


In [ ]:
def p_mle(prev, w):
    return bigram[(prev, w)] / unigram[prev] if unigram[prev] else 0.0

def p_laplace(prev, w):
    return (bigram[(prev, w)] + 1) / (unigram[prev] + V)

def p_addk(prev, w, k):
    return (bigram[(prev, w)] + k) / (unigram[prev] + k * V)

def p_unigram(w):
    return (uni_pred[w] + 1) / (N_pred + V)

def p_interp(prev, w, lam):
    return lam * p_mle(prev, w) + (1 - lam) * p_unigram(w)

def sentence_table(sent):
    p = prep(sent)
    rows = []
    for prev, w in zip(p[:-1], p[1:]):
        c_bi, c_prev = bigram[(prev, w)], unigram[prev]
        rows.append({
            'bigram P(w | prev)': f'P({w} | {prev})',
            'C(prev,w)': c_bi, 'C(prev)': c_prev,
            'MLE': c_bi / c_prev if c_prev else 0.0,
            'Laplace = (C+1)/(C(prev)+V)': (c_bi + 1) / (c_prev + V),
        })
    return pd.DataFrame(rows)

def show_calculation(sent):

    print('Sentence (after preprocessing):', sent)
    print('With <UNK> mapping + markers  :', prep(sent), '\n')
    tbl = sentence_table(sent)
    display(tbl)
    mle = tbl['MLE'].tolist()
    lap = tbl['Laplace = (C+1)/(C(prev)+V)'].tolist()
    print('\nMarkov assumption:  P(sentence) ≈ ' + ' × '.join(tbl['bigram P(w | prev)']))
    print('\nMLE      :', ' × '.join(f'{x:.3e}' for x in mle))
    print('         = ', math.prod(mle), '   <-- 0 as soon as ONE bigram is unseen')
    print('\nLaplace  :', ' × '.join(f'{x:.3e}' for x in lap))
    print('         = ', math.prod(lap))
    print('\nlog10 P (Laplace) = Σ log10(term) =', sum(math.log10(x) for x in lap),
          '  (logs avoid underflow for long sentences)')

pd.set_option('display.float_format', lambda x: f'{x:.3e}' if (x != 0 and abs(x) < 1e-3) else f'{x:.6f}')

 sentence from the **TEST** set

In [ ]:
for raw in test_df['text']:
    ps = preprocess(raw)
    if ps and 5 <= len(ps[0]) <= 7:
        break
print('Raw sentence :', sent_tokenize(fix_hyphens(raw))[0], '\n')
show_calculation(ps[0])

Raw sentence : Ky. Company Wins Grant to Study Peptides. 

Sentence (after preprocessing): ['company', 'win', 'grant', 'study', 'peptide']
With <UNK> mapping + markers  : ['<s>', 'company', 'win', 'grant', 'study', 'peptide', '</s>'] 



,bigram P(w | prev),"C(prev,w)",C(prev),MLE,Laplace = (C+1)/(C(prev)+V)
0,P(company | <s>),584,245400,0.002380,0.002105
1,P(win | company),6,9170,6.543e-04,1.677e-04
2,P(grant | win),0,6050,0.000000,2.590e-05
3,P(study | grant),3,300,0.010000,1.217e-04
4,P(peptide | study),0,1305,0.000000,2.952e-05
5,P(</s> | peptide),2,5,0.400000,9.210e-05



Markov assumption:  P(sentence) ≈ P(company | <s>) × P(win | company) × P(grant | win) × P(study | grant) × P(peptide | study) × P(</s> | peptide)

MLE      : 2.380e-03 × 6.543e-04 × 0.000e+00 × 1.000e-02 × 0.000e+00 × 4.000e-01
         =  0.0    <-- 0 as soon as ONE bigram is unseen

Laplace  : 2.105e-03 × 1.677e-04 × 2.590e-05 × 1.217e-04 × 2.952e-05 × 9.210e-05
         =  3.024801077960176e-24

log10 P (Laplace) = Σ log10(term) = -23.51930318087427   (logs avoid underflow for long sentences)


In [ ]:
show_calculation(train_sents[0])

Sentence (after preprocessing): ['carlyle', 'look', 'toward', 'commercial', 'aerospace']
With <UNK> mapping + markers  : ['<s>', 'carlyle', 'look', 'toward', 'commercial', 'aerospace', '</s>'] 



,bigram P(w | prev),"C(prev,w)",C(prev),MLE,Laplace = (C+1)/(C(prev)+V)
0,P(carlyle | <s>),10,245400,4.075e-05,3.957e-05
1,P(look | carlyle),1,16,0.062500,6.138e-05
2,P(toward | look),4,2492,0.001605,1.426e-04
3,P(commercial | toward),1,689,0.001451,6.014e-05
4,P(aerospace | commercial),1,479,0.002088,6.052e-05
5,P(</s> | aerospace),3,108,0.027778,1.224e-04



Markov assumption:  P(sentence) ≈ P(carlyle | <s>) × P(look | carlyle) × P(toward | look) × P(commercial | toward) × P(aerospace | commercial) × P(</s> | aerospace)

MLE      : 4.075e-05 × 6.250e-02 × 1.605e-03 × 1.451e-03 × 2.088e-03 × 2.778e-02
         =  3.440805915581632e-16    <-- 0 as soon as ONE bigram is unseen

Laplace  : 3.957e-05 × 6.138e-05 × 1.426e-04 × 6.014e-05 × 6.052e-05 × 1.224e-04
         =  1.5435628899737003e-25

log10 P (Laplace) = Σ log10(term) = -24.811475671195083   (logs avoid underflow for long sentences)


In [ ]:
def next_word_probs(prev, k=8):
    cands = sorted(followers[prev], key=lambda x: -x[1])[:k]
    return pd.DataFrame([{'next word': w, 'C(prev,w)': c, 'C(prev)': unigram[prev],
                          'P(w|prev) MLE': c / unigram[prev]} for w, c in cands])

for w in ['oil', 'microsoft']:
    print(f'\nMost likely words after "{w}":')
    display(next_word_probs(w))


Most likely words after "oil":


,next word,"C(prev,w)",C(prev),P(w|prev) MLE
0,price,2659,6782,0.392067
1,giant,216,6782,0.031849
2,company,212,6782,0.031259
3,future,203,6782,0.029932
4,</s>,147,6782,0.021675
5,gas,113,6782,0.016662
6,rich,109,6782,0.016072
7,supply,106,6782,0.015630



Most likely words after "microsoft":


,next word,"C(prev,w)",C(prev),P(w|prev) MLE
0,</s>,210,5368,0.039121
1,window,121,5368,0.022541
2,say,119,5368,0.022168
3,corp,110,5368,0.020492
4,release,96,5368,0.017884
5,launch,79,5368,0.014717
6,internet,78,5368,0.014531
7,announce,62,5368,0.011550


## 4. Evaluation — perplexity

$$PP(W)=\exp\!\Big(-\frac{1}{N}\sum_{i=1}^{N}\ln P(w_i\mid w_{i-1})\Big)$$

Lower perplexity = better model. All models are **trained on train only**.

* The hyper-parameters (`K` for add-k, `λ` for interpolation) are chosen on the **validation** set.
* The **test** set is used only once, for the final numbers.

In [ ]:
def perplexity(sents, prob_fn):
    ls, n = 0.0, 0
    for s in sents:
        p = prep(s)
        for prev, w in zip(p[:-1], p[1:]):
            ls += math.log(prob_fn(prev, w)); n += 1
    return math.exp(-ls / n), n

k_grid   = [1, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001, 0.0005, 0.0001]
lam_grid = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

val_k   = {k: perplexity(val_sents, lambda a, b, k=k: p_addk(a, b, k))[0] for k in k_grid}
val_lam = {l: perplexity(val_sents, lambda a, b, l=l: p_interp(a, b, l))[0] for l in lam_grid}
best_k   = min(val_k,   key=val_k.get)
best_lam = min(val_lam, key=val_lam.get)

print('Validation perplexity, add-k        :', {k: round(v, 1) for k, v in val_k.items()})
print('Validation perplexity, interpolation:', {l: round(v, 1) for l, v in val_lam.items()})
print(f'\nChosen on validation: K = {best_k}, lambda = {best_lam}')

Validation perplexity, add-k        : {1: 4442.1, 0.5: 3112.1, 0.1: 1465.3, 0.05: 1147.3, 0.01: 841.2, 0.005: 818.2, 0.001: 921.5, 0.0005: 1025.2, 0.0001: 1410.5}
Validation perplexity, interpolation: {0.5: 529.5, 0.6: 513.2, 0.7: 511.5, 0.8: 529.8, 0.9: 593.2, 0.95: 684.6}

Chosen on validation: K = 0.005, lambda = 0.7


In [ ]:

results = {
    'Unigram (add-one)':                          perplexity(test_sents, lambda a, b: p_unigram(b)),
    'Bigram (add-one / Laplace)':                 perplexity(test_sents, p_laplace),
    f'Bigram add-k (k={best_k})':                 perplexity(test_sents, lambda a, b: p_addk(a, b, best_k)),
    f'Bigram + unigram interpolation (λ={best_lam})': perplexity(test_sents, lambda a, b: p_interp(a, b, best_lam)),
}
n_pred = next(iter(results.values()))[1]
print(f'Test predictions evaluated: {n_pred:,}\n')
display(pd.DataFrame({'Test perplexity': {k: v[0] for k, v in results.items()}}).round(1))

Test predictions evaluated: 201,012



,Test perplexity
Unigram (add-one),2407.400000
Bigram (add-one / Laplace),4414.600000
Bigram add-k (k=0.005),822.500000
Bigram + unigram interpolation (λ=0.7),515.100000


In [ ]:
unseen, total = 0, 0
for s in test_sents:
    p = prep(s)
    for prev, w in zip(p[:-1], p[1:]):
        total += 1
        unseen += bigram[(prev, w)] == 0
print(f'Unseen bigrams in test: {unseen:,} / {total:,} = {unseen/total:.1%}')

Unseen bigrams in test: 49,119 / 201,012 = 24.4%


## 5. Summary

* **Preprocessing:** cleaning (HTML/URL/agency boilerplate) → sentence segmentation → tokenization → POS tagging → lowercasing → punctuation/number removal → stop-word removal → POS-aware lemmatization.
* **Data split:** 90 % of train builds the vocabulary and the unigram/bigram counts; 10 % of train (validation) only selects `K` and `λ`; the test set is used only for the worked unseen-sentence example and the final perplexity.
* **Markov assumption:** $P(w_i \mid w_1..w_{i-1}) \approx P(w_i \mid w_{i-1})$, so $P(sentence) \approx \prod P(w_i \mid w_{i-1})$ with `<s>`/`</s>` padding.
* **Smoothing:** without smoothing, any unseen bigram makes the whole sentence probability 0; add-one, add-k and interpolation are compared by perplexity.
* **Note:** stop words are removed *before* building the bigrams (as the assignment requires), so two words that are neighbours in the model were not always neighbours in the original sentence.